# Supervised Learning

Choose **one regression task and one classification task**. For each, examine the data, justify preprocessing, compare models, and explain the results.
Assessment considers performance on unseen test data, training error, and the quality of the report.

## 1. Prediction tasks

Each dataset contains **5,000 labeled observations**. Feature counts exclude identifiers and targets.
See the corresponding `data_dictionary.md` for individual column definitions.

### Robot joint torque · Real regression

**Dataset:** `sarcos_regression`.

**Target:** `torque_1`, the torque at the robot's first joint, on the original numerical scale.

**Inputs (21):** Seven joint positions, seven velocities, and seven accelerations.

**Remark:** Feature scales differ; consider their effect on regularized models.  
Negative and extreme values may describe valid operating conditions.  
Physical units are incompletely documented; report errors on the source scale.

Source: [SARCOS](https://gaussianprocess.org/gpml/data/).

### Building cooling load · Synthetic regression

**Dataset:** `cooling_regression`.

**Target:** `cooling_load_kw`, the total thermal cooling load of four building zones, in kW.

**Inputs (20):** Four zones, each with area, temperature difference, heat-transfer coefficient, solar irradiance, and internal heat gain.

**Remark:** Temperature differences are outdoor minus indoor; some observations are missing.  
Missing temperatures are not zeros, while zero solar irradiance is valid.  
The simplified simulation describes thermal cooling load, not electrical power consumption.

### Credit card default · Real classification

**Dataset:** `credit_classification`.

**Target:** `default_next_month`: 1 means recorded payment default in the following month; 0 means no default.
Default means failing to meet repayment obligations. Predict the **probability of class 1**.

**Inputs (20):** Credit limit, age, and six months each of repayment status, bill amounts, and payment amounts.

**Remark:** Repayment status codes are not necessarily equally spaced; justify their encoding.  
Negative codes and bill amounts can be valid; check before transforming them.  
Examine class balance; these records describe customers in 2005.

Source: [UCI credit card default](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients).

### Order execution risk · Synthetic classification

**Dataset:** `execution_classification`.

**Target:** `high_slippage`: 1 means adverse execution slippage exceeds 5 basis points (0.05%); 0 means otherwise.
Predict the **probability of class 1**.

**Inputs (20):** Bid/ask prices and quantities at four levels, plus order size, direction, execution delay, and recent volatility.

**Remark:** Buys consume asks; sells consume bids, and `side=-1` is valid.  
Slippage is measured from the best executable quote at submission, not the mid-price.  
Future market changes are random; the simulation is not calibrated to a real market.

## 2. Required experiments

Complete the following for **both selected tasks**, except where stated otherwise.

| Experiment | What to do |
|---|---|
| Data diagnosis | Examine feature meanings, ranges, missingness, skewness, and unusual values. For classification, also examine class proportions. Justify numerical or categorical treatment. |
| Preprocessing comparison | Compare two versions that differ in one justified preprocessing choice. Keep the learner, folds, and tuning budget the same; report the difference for each fold. |
| Baseline | Use Linear Regression or Ridge for regression, and Logistic Regression for classification. Explain preprocessing and parameter choices. |
| Bagging | Fit the baseline with the same feature representation on bootstrap samples and average predictions or class-1 probabilities. Record the number of models and other settings. |
| Improvement (optional) | Add justified features or use another model. Explain the change and compare it fairly with the baseline. |
| Learning curve | For **at least one task**, compare training and validation error at **three or more training sizes**. Subsample only the training portion of each fold; keep validation rows fixed. |

Possible preprocessing comparisons include scaling for robot torque, imputation for cooling load, categorical encoding of repayment status, and relative prices for execution risk.
Keep other necessary preprocessing in both comparison conditions.

Explain why each change helped, had little effect, or reduced performance. Bagging need not improve a model that underfits. Describe any changes to regularization or other baseline settings separately.
For the learning curve, discuss what the results suggest about model complexity, sample size, and noise; preserve class proportions when subsampling classification data.

## 3. Setup and data

Place this notebook, `train.py`, `test.py`, and the existing `datasets/` folder in the same working directory.
This layout works in local Jupyter or Colab; in Colab, upload the files and extract the data archive first.

- Edit `make_model` in **`train.py`** to define preprocessing, models, and hyperparameters.
- Edit `predict` in **`test.py`** only if the default prediction method does not support your model. Leave metric calculations unchanged.
- Use this notebook for experiments and analysis. Do not copy the two scripts into notebook cells.

Run the example for one dataset at a time. Change `DATASET` and repeat for your other selected task.

In [ ]:
# In Colab, upload train.py, test.py, and datasets.zip.
# In local Jupyter, place these files beside this notebook.
try:
    from google.colab import files
except ImportError:
    print("Local Jupyter: using the files in the working directory.")
else:
    uploaded = files.upload()


In [ ]:
# Extract the supplied dataset archive if the folder is not already present.
from pathlib import Path
from zipfile import ZipFile

if not Path("datasets").is_dir():
    with ZipFile("datasets.zip") as archive:
        archive.extractall(".")
print("Dataset folder:", Path("datasets").resolve())


In [ ]:

%pip install -q "numpy>=1.26" "pandas>=2.1" "scikit-learn>=1.4,<2" "matplotlib>=3.7"
from pathlib import Path
import importlib
import json
import pandas as pd
import train as training
import test as testing

DATA_ROOT = Path("datasets")
DATASET = "sarcos_regression"  # 先做sarcos_regression，然后换成execution_classification
SEED = 2026
# Setup completed.

In [ ]:
data_dir = DATA_ROOT / DATASET
metadata = json.loads((data_dir / "metadata.json").read_text())
data = pd.read_csv(data_dir / "train.csv", dtype={"sample_id": str})
X = data[metadata["feature_names"]]
y = data[metadata["target"]]

print(f"{len(data)} observations, {X.shape[1]} features; target: {metadata['target']}")
display(data.head())
feature_summary = X.describe().T
feature_summary["missing_rate"] = X.isna().mean()
feature_summary["skewness"] = X.skew(numeric_only=True)
display(feature_summary)
if metadata["task"] == "binary_classification":
    display(y.value_counts(normalize=True).sort_index().rename("class_fraction"))
else:
    display(y.describe().rename("target_summary"))
# Data loaded


## 4. Training and validation

Use the supplied **five-fold cross-validation** assignments in `cv_folds.csv`.
The 5,000 observations are divided into five groups of 1,000. Train on four groups and validate on the remaining group, rotating as follows:

| Run | Training folds | Validation fold |
|---|---|---|
| 1 | 1, 2, 3, 4 | 0 |
| 2 | 0, 2, 3, 4 | 1 |
| 3 | 0, 1, 3, 4 | 2 |
| 4 | 0, 1, 2, 4 | 3 |
| 5 | 0, 1, 2, 3 | 4 |

Each run uses **4,000 training rows and 1,000 validation rows**. Fit a fresh model and fit all learned preprocessing on the training rows only.
Keep the same folds when comparing methods. Neither `sample_id` nor `fold` is a model input.
The separate held-out test set is reserved for final evaluation.

The next cell checks the supplied folds and prepares a directory for experiment scores. `training.validate()` aligns folds by `sample_id` and fits a fresh model for each split.

In [ ]:
folds = pd.read_csv(data_dir / "cv_folds.csv", dtype={"sample_id": str})
assert data["sample_id"].is_unique and folds["sample_id"].is_unique
assert set(data["sample_id"]) == set(folds["sample_id"])
fold_id = data["sample_id"].map(folds.set_index("sample_id")["fold"])
assert set(fold_id) == set(range(5))

run_dir = Path("runs") / DATASET
run_dir.mkdir(parents=True, exist_ok=True)

`train(data_path)` returns a fitted model bundle, including preprocessing.
`test(data_path, model)` returns a report whose `metrics` field contains the scores.
The example below runs the supplied baseline. After implementing another method in `train.py`, change `METHOD` and rerun this cell.

| Task | Main metric | Also report | Additional metrics computed automatically |
|---|---|---|---|
| Regression | RMSE | MAE | MSE, R² |
| Classification | Log loss | ROC-AUC | Accuracy, F1 |

Classification models output class-1 probabilities.
Accuracy and F1 use threshold 0.5; log loss clips probabilities to `[1e-15, 1 - 1e-15]`.

In [ ]:
importlib.reload(training)  # Pick up edits to train.py and test.py.
importlib.reload(testing)
METHOD = "baseline"  # Change after implementing another method.

results, summary = training.validate(data_dir / "train.csv", method=METHOD, seed=SEED)
results.to_csv(run_dir / f"{METHOD}_scores.csv", index=False)
display(results)
display(summary)

Add the preprocessing comparison, other models, and learning curve below, with plots and explanations.
The fold scores for each method are saved under `runs/`; retain them when comparing methods or changing parameters.

In [ ]:
# Compare preprocessing and Bagging for the current dataset.
import matplotlib.pyplot as plt

score_tables = {METHOD: results} # 先放入baseline
for method_name in ["unscaled", "bagging"]:
    fold_scores, fold_summary = training.validate(data_dir / "train.csv", method=method_name, seed=SEED)
    fold_scores.to_csv(run_dir / f"{method_name}_scores.csv", index=False)
    score_tables[method_name] = fold_scores # 放入对比模型

    display(fold_scores)
    display(fold_summary)

# sarcos_regression做learning curve
if DATASET == "sarcos_regression":
    curve_results = []
    # 总数据5k，选择5种不同的训练集大小，分别为500, 1000, 2000, 3000, 4000
    for n_train in [500, 1000, 2000, 3000, 4000]:
        fold_scores, _ = training.validate(data_dir / "train.csv", method="baseline", seed=SEED, train_size=n_train)
        curve_results.append(fold_scores)

    # 合成一张表
    learning_curve = pd.concat(curve_results, ignore_index=True)
    learning_curve.to_csv(run_dir / "learning_curve_scores.csv", index=False)
    learning_summary = learning_curve.groupby("n_train")[["train_rmse", "validation_rmse"]].agg(["mean", "std"])
    display(learning_summary)

    # 画图
    ax = learning_curve.groupby("n_train")[["train_rmse", "validation_rmse"]].mean().plot(marker="o")
    ax.set(xlabel="Training rows per fold", ylabel="RMSE (source scale)",
           title="SARCOS Ridge learning curve")
    ax.figure.tight_layout()
    ax.figure.savefig(run_dir / "learning_curve_rmse.png", dpi=160)
    plt.show()

main_metric = "rmse" if metadata["task"] == "regression" else "log_loss"
selected_method = min(
    score_tables,
    key=lambda name: score_tables[name][f"validation_{main_metric}"].mean()
)
print(f"Selected method for {DATASET}: {selected_method}")


**Experiment settings**

All methods use seed 2026 and the supplied five folds. Baseline: median imputation, StandardScaler, Ridge (`alpha=1`) for regression or Logistic Regression (`C=1`, `max_iter=2000`) for classification. `unscaled` changes only the scaler. Bagging uses 100 bootstrap fits of the baseline learner.

## SARCOS regression

**Data diagnosis**

Rows: 5,000  
Features: 21 (7 positions, 7 velocities, 7 accelerations)  
Missing values: 0  
Feature ranges: positions -0.923 to 2.249; velocities -4.734 to 4.683; accelerations -47.764 to 70.845  
Skew/unusual values: `position_6` skewness -1.67; negative and extreme operating values are valid and retained  
Target: `torque_1`, -90.39 to 112.27; mean 13.26; errors use the original scale

**Five-fold results** (mean ± sample SD)

| Method | Train RMSE | Validation RMSE | Validation MAE | Validation R² |
|---|---:|---:|---:|---:|
| Baseline | 5.4912 ± 0.0277 | 5.5305 ± 0.1083 | 3.9639 ± 0.0923 | 0.9265 ± 0.0051 |
| Unscaled | 5.4914 ± 0.0277 | 5.5303 ± 0.1083 | 3.9629 ± 0.0914 | 0.9265 ± 0.0052 |
| Bagging | 5.4913 ± 0.0277 | 5.5303 ± 0.1079 | 3.9638 ± 0.0939 | 0.9265 ± 0.0051 |

**Preprocessing comparison by fold** (delta = unscaled − baseline; lower RMSE is better)

| Fold | Baseline RMSE | Unscaled RMSE | Delta |
|---:|---:|---:|---:|
| 0 | 5.543601 | 5.545206 | +0.001605 |
| 1 | 5.624360 | 5.623460 | -0.000901 |
| 2 | 5.471403 | 5.473398 | +0.001994 |
| 3 | 5.377460 | 5.375349 | -0.002111 |
| 4 | 5.635466 | 5.633868 | -0.001598 |

Scaling has virtually no effect: the small differences change direction across folds. Bagging also has virtually no effect for this Ridge setting.

**Learning curve** (mean ± sample SD across folds; validation rows fixed)

| Training rows | Train RMSE | Validation RMSE |
|---:|---:|---:|
| 500 | 5.2722 ± 0.1729 | 5.7050 ± 0.1256 |
| 1,000 | 5.4015 ± 0.1865 | 5.5761 ± 0.0888 |
| 2,000 | 5.4342 ± 0.1099 | 5.5548 ± 0.1140 |
| 3,000 | 5.4981 ± 0.0108 | 5.5471 ± 0.1020 |
| 4,000 | 5.4912 ± 0.0277 | 5.5305 ± 0.1083 |

Validation RMSE improves as training size grows, with smaller gains after about 1,000–2,000 rows. The train-validation gap narrows from about 0.43 at 500 rows to 0.04 at 4,000, suggesting more data helps this fixed Ridge model. The conclusion is limited to `alpha=1` and these folds. The plot is saved as `runs/sarcos_regression/learning_curve_rmse.png`.

**Selected method / full-development training score**: `unscaled`; RMSE 5.4953, MAE 3.9382, R² 0.9278.

## Execution classification

**Data diagnosis**

Rows: 5,000  
Features: 20 (four levels of bid/ask prices and quantities, order size, side, latency, volatility)  
Missing values: 0  
Feature ranges: prices 24.99 to 200.09; quantities and order size 25 to 4,365  
Skewness: `order_size` 0.93; price features about 0.71  
Side: -1 = sell (2,488 rows); +1 = buy (2,512 rows); both values are valid  
Target: class 0 = 3,608 (72.16%); class 1 = 1,392 (27.84%); class imbalance makes probability metrics useful

**Five-fold results** (mean ± sample SD)

| Method | Train log loss | Validation log loss | Validation ROC-AUC | Validation accuracy | Validation F1 |
|---|---:|---:|---:|---:|---:|
| Baseline | 0.5262 ± 0.0024 | 0.5291 ± 0.0099 | 0.7265 ± 0.0146 | 0.7218 ± 0.0085 | 0.2689 ± 0.0248 |
| Unscaled | 0.5243 ± 0.0024 | 0.5273 ± 0.0099 | 0.7289 ± 0.0146 | 0.7240 ± 0.0078 | 0.2781 ± 0.0237 |
| Bagging | 0.5261 ± 0.0024 | 0.5289 ± 0.0096 | 0.7267 ± 0.0143 | 0.7218 ± 0.0090 | 0.2712 ± 0.0280 |

**Preprocessing comparison by fold** (delta = unscaled − baseline; lower log loss is better)

| Fold | Baseline log loss | Unscaled log loss | Delta |
|---:|---:|---:|---:|
| 0 | 0.535485 | 0.533529 | -0.001956 |
| 1 | 0.513054 | 0.510817 | -0.002237 |
| 2 | 0.535559 | 0.533700 | -0.001860 |
| 3 | 0.526015 | 0.525111 | -0.000904 |
| 4 | 0.535377 | 0.533586 | -0.001791 |

Unscaled inputs reduce log loss in all five folds, but the average gain is small (about 0.00175). Bagging changes mean validation log loss from 0.5291 to 0.5289, so it gives little benefit for this Logistic Regression setting.

**Selected method / full-development training score**: `unscaled`; log loss 0.5244, ROC-AUC 0.7334, accuracy 0.7238, F1 0.2796.

These full-development scores are training scores, not held-out test results. The test data have not been provided.


## 5. Train the selected model

After model selection, fit the chosen configuration on all 5,000 development observations. Keep the returned model bundle in memory for evaluation when the separate test data are provided. Running the notebook for the other selected task creates that task's model in the same way.


In [ ]:
FINAL_METHOD = selected_method
importlib.reload(training)
model = training.train(data_dir / "train.csv", method=FINAL_METHOD, seed=SEED)
print(f"Trained {DATASET} ({FINAL_METHOD}) on all {len(data):,} development rows.")


For a separate labeled test CSV, pass the model bundle in the same Python session:

```python
report = testing.test("path/to/test.csv", model)
print(report["metrics"])
```

No preprocessing or model fitting takes place during testing. Without true labels, error cannot be calculated. After a runtime restart, rerun training to recreate the model bundle.


## 6. Report and submission

Submit the completed notebook or report, `train.py`, `test.py`, any additional modules, and instructions for reproducing the results.
Identify both selected datasets and document dependencies, random seeds, preprocessing, features, and hyperparameters.
Include data diagnosis, preprocessing and model comparisons, a learning curve, and an interpretation of the results.
Final test administration will be specified separately.

Use only the supplied data for training. Do not retrieve public-source labels to match records or obtain held-out answers.
Cite external resources used.

**AI policy.** AI tools are permitted, but disclose the tools and how you used them—for example, model implementation or deployment,
hyperparameter search, data analysis, or report drafting and editing. Identify the affected code, experiments, or report sections.
If you did not use AI, state this. You remain responsible for the correctness of the work and for explaining it.

**Readability affects the grade.** Organize the report logically, use consistent notation, label figures and tables, and explain what the results show.

**AI Use Disclosure:** _State the tools used and their contributions, or state that no AI tools were used._
